In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from Bio import PDB

def read_protein_atoms(pdb_file):
    """
    Reads a PDB file and extracts atom coordinates.
    Returns a list of (atom_name, x, y, z) tuples.
    """
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    
    protein_atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atom_name = atom.get_name()  # Atom type (e.g., CA, CB, O, N)
                    x, y, z = atom.get_coord()  # Extract X, Y, Z coordinates
                    protein_atoms.append((atom_name, x, y, z))
    
    return protein_atoms

def read_ligand_atoms(sdf_file):
    """
    Reads an SDF file and extracts atom coordinates.
    Returns a list of (atom_name, x, y, z) tuples.
    """
    suppl = Chem.SDMolSupplier(sdf_file, removeHs=False)
    
    ligand_atoms = []
    for mol in suppl:
        if mol is None:
            continue
        
        conf = mol.GetConformer()
        for atom in mol.GetAtoms():
            atom_name = atom.GetSymbol()  # Element type (C, O, N, etc.)
            pos = conf.GetAtomPosition(atom.GetIdx())  # Get 3D coordinates
            ligand_atoms.append((atom_name, pos.x, pos.y, pos.z))
    
    return ligand_atoms

def calculate_distances(protein_atoms, ligand_atoms):
    """
    Computes Euclidean distances between each protein and ligand atom.
    Returns a Pandas DataFrame with columns: Protein Atom, Ligand Atom, Distance.
    """
    data = []
    
    for p_atom, p_x, p_y, p_z in protein_atoms:
        for l_atom, l_x, l_y, l_z in ligand_atoms:
            distance = np.sqrt((p_x - l_x)**2 + (p_y - l_y)**2 + (p_z - l_z)**2)
            data.append([p_atom, l_atom, round(distance, 3)])  # Round to 3 decimals

    df = pd.DataFrame(data, columns=["Protein Atom", "Ligand Atom", "Distance (Å)"])
    return df

# ------------------ MAIN EXECUTION ------------------
pdb_file = r"E:\NewProject\example_ref\1a1e\1a1e_protein.pdb"   # Change to your PDB file path
sdf_file = r"E:\NewProject\example_ref\1a1e\1a1e_ligandMS.sdf"    # Change to your SDF file path

# Read atoms from files
protein_atoms = read_protein_atoms(pdb_file)
ligand_atoms = read_ligand_atoms(sdf_file)

# Compute distances
df_distances = calculate_distances(protein_atoms, ligand_atoms)

# Save to CSV
df_distances.to_csv("protein_ligand_distances.csv", index=False)
print("Distance calculations complete! Results saved to protein_ligand_distances.csv")


Distance calculations complete! Results saved to protein_ligand_distances.csv


In [2]:
# Display first few rows
print(df_distances.head())

  Protein Atom Ligand Atom  Distance (Å)
0            N           C     41.529999
1            N           O     40.646999
2            N           C     41.553001
3            N           N     42.414001
4            N           C     42.509998


In [3]:
import numpy as np
import pandas as pd
from rdkit import Chem
from Bio import PDB

def read_protein_atoms(pdb_file):
    """
    Reads a PDB file and extracts atom coordinates along with residue names.
    Returns a list of (residue_name, atom_name, x, y, z) tuples.
    """
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    
    protein_atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                res_name = residue.get_resname()  # Residue name (e.g., ALA, TYR)
                for atom in residue:
                    atom_name = atom.get_name()  # Atom type (e.g., CA, CB, O, N)
                    x, y, z = atom.get_coord()  # Extract X, Y, Z coordinates
                    protein_atoms.append((res_name, atom_name, x, y, z))
    
    return protein_atoms

def read_ligand_atoms(sdf_file):
    """
    Reads an SDF file and extracts atom coordinates.
    Returns a list of (atom_name, x, y, z) tuples.
    """
    suppl = Chem.SDMolSupplier(sdf_file, removeHs=False)
    
    ligand_atoms = []
    for mol in suppl:
        if mol is None:
            continue
        
        conf = mol.GetConformer()
        for atom in mol.GetAtoms():
            atom_name = atom.GetSymbol()  # Element type (C, O, N, etc.)
            pos = conf.GetAtomPosition(atom.GetIdx())  # Get 3D coordinates
            ligand_atoms.append((atom_name, pos.x, pos.y, pos.z))
    
    return ligand_atoms

def calculate_distances(protein_atoms, ligand_atoms, distance_cutoff=6.0):
    """
    Computes Euclidean distances between each protein and ligand atom.
    Returns a Pandas DataFrame with columns: Residue Name, Protein Atom, Ligand Atom, Distance.
    Only keeps distances ≤ distance_cutoff (default: 6 Å).
    """
    data = []
    
    for res_name, p_atom, p_x, p_y, p_z in protein_atoms:
        for l_atom, l_x, l_y, l_z in ligand_atoms:
            distance = np.sqrt((p_x - l_x)**2 + (p_y - l_y)**2 + (p_z - l_z)**2)
            
            if distance <= distance_cutoff:  # Only keep distances ≤ 6 Å
                data.append([res_name, p_atom, l_atom, round(distance, 3)])

    df = pd.DataFrame(data, columns=["Residue", "Protein Atom", "Ligand Atom", "Distance (Å)"])
    return df

# ------------------ MAIN EXECUTION ------------------
pdb_file = r"E:\NewProject\example_ref\1a1e\1a1e_protein.pdb"   # Change to your PDB file path
sdf_file = r"E:\NewProject\example_ref\1a1e\1a1e_ligandMS.sdf" 

# Read atoms from files
protein_atoms = read_protein_atoms(pdb_file)
ligand_atoms = read_ligand_atoms(sdf_file)

# Compute distances with a cutoff of 6 Å
df_distances = calculate_distances(protein_atoms, ligand_atoms, distance_cutoff=6.0)

# Save to CSV
df_distances.to_csv("protein_ligand_distances_filtered.csv", index=False)
print("Distance calculations complete! Results saved to protein_ligand_distances_filtered.csv")


Distance calculations complete! Results saved to protein_ligand_distances_filtered.csv


In [4]:
# Display first few rows
print(df_distances.head())

  Residue Protein Atom Ligand Atom  Distance (Å)
0     ILE            O           O         5.963
1     ILE            O           H         5.522
2     THR           CA           H         5.987
3     THR           HA           O         5.350
4     THR           HA           H         5.209


In [5]:
df_distances.shape

(2407, 4)

In [6]:
import numpy as np
import pandas as pd
from rdkit import Chem
from Bio import PDB
import itertools

# **Define 1540 ECIF Atom Pairs** (Load from ECIF reference)
ECIF_PAIRS = [
    ("C;4;1;3;0;0", "C;4;3;0;0;1")  # Example, continue for all 1540 pairs
]  

def read_protein_atoms(pdb_file):
    """Extracts protein atoms, residue names, and ECIF types from a PDB file."""
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    
    protein_atoms = []
    for model in structure:
        for chain in model:
            for residue in chain:
                res_name = residue.get_resname()  # Residue name (ALA, TYR, etc.)
                for atom in residue:
                    atom_name = atom.element  # Element (C, N, O, etc.)
                    x, y, z = atom.get_coord()
                    protein_atoms.append((res_name, atom_name, x, y, z))
    
    return protein_atoms

def read_ligand_atoms(sdf_file):
    """Extracts ligand atoms and their ECIF types from an SDF file."""
    suppl = Chem.SDMolSupplier(sdf_file, removeHs=False)
    
    ligand_atoms = []
    for mol in suppl:
        if mol is None:
            continue
        conf = mol.GetConformer()
        for atom in mol.GetAtoms():
            atom_name = atom.GetSymbol()  # Element type (C, O, N, etc.)
            pos = conf.GetAtomPosition(atom.GetIdx())
            ligand_atoms.append((atom_name, pos.x, pos.y, pos.z))
    
    return ligand_atoms

def calculate_distances(protein_atoms, ligand_atoms, distance_cutoff=6.0):
    """Computes distances for ECIF atom pairs within the cutoff."""
    data = []
    
    for res_name, p_atom, p_x, p_y, p_z in protein_atoms:
        for l_atom, l_x, l_y, l_z in ligand_atoms:
            distance = np.sqrt((p_x - l_x)**2 + (p_y - l_y)**2 + (p_z - l_z)**2)

            # Find the matching ECIF pair for this atom combination
            for ecif_p, ecif_l in ECIF_PAIRS:
                if f"{p_atom}-{l_atom}" in [ecif_p.split(";")[0] + "-" + ecif_l.split(";")[0]]:
                    if distance <= distance_cutoff:
                        ecif_pair = f"{ecif_p}-{ecif_l}"
                        data.append([res_name, ecif_pair, round(distance, 3)])

    df = pd.DataFrame(data, columns=["Residue", "Protein-ligand_ECIF_pair", "Distance(Å)"])
    return df

# ------------------ MAIN EXECUTION ------------------
pdb_file = r"E:\NewProject\example_ref\1a1e\1a1e_protein.pdb"   # Change to your PDB file path
sdf_file = r"E:\NewProject\example_ref\1a1e\1a1e_ligandMS.sdf" 

# Read atoms from files
protein_atoms = read_protein_atoms(pdb_file)
ligand_atoms = read_ligand_atoms(sdf_file)

# Compute distances with a cutoff of 6 Å
df_distances = calculate_distances(protein_atoms, ligand_atoms, distance_cutoff=6.0)

# Save to CSV
df_distances.to_csv("protein_ligand_ecif_distances.csv", index=False)
print("Distance calculations complete! Results saved to protein_ligand_ecif_distances.csv")

Distance calculations complete! Results saved to protein_ligand_ecif_distances.csv


In [7]:
# Display first few rows
print(df_distances.head())

  Residue Protein-ligand_ECIF_pair  Distance(Å)
0     ARG  C;4;1;3;0;0-C;4;3;0;0;1        5.857
1     ARG  C;4;1;3;0;0-C;4;3;0;0;1        3.779
2     ARG  C;4;1;3;0;0-C;4;3;0;0;1        3.753
3     ARG  C;4;1;3;0;0-C;4;3;0;0;1        5.303
4     ARG  C;4;1;3;0;0-C;4;3;0;0;1        5.948


In [8]:
import pandas as pd

# Load the existing distances CSV (generated from the previous script)
df_distances = pd.read_csv("protein_ligand_ecif_distances.csv")

def count_ecif_pairs(df, distance_cutoff):
    """
    Counts occurrences of each Protein-ligand_ECIF_pair for each Residue within the given distance cutoff.
    """
    # Filter for the specified distance cutoff
    df_filtered = df[df["Distance(Å)"] <= distance_cutoff]

    # Group by Residue and Protein-ligand_ECIF_pair, then count occurrences
    df_counts = df_filtered.groupby(["Residue", "Protein-ligand_ECIF_pair"]).size().reset_index(name="Count")

    return df_counts

# Example: Count occurrences within 6.0 Å cutoff
distance_cutoff = 6.0
df_counts = count_ecif_pairs(df_distances, distance_cutoff)

# Save to CSV
df_counts.to_csv("ecif_pair_counts_cutoff_6A.csv", index=False)

In [9]:
# Display top rows
print(df_counts.head())

  Residue Protein-ligand_ECIF_pair  Count
0     ARG  C;4;1;3;0;0-C;4;3;0;0;1     13
1     ASP  C;4;1;3;0;0-C;4;3;0;0;1      2
2     CYS  C;4;1;3;0;0-C;4;3;0;0;1      6
3     GLY  C;4;1;3;0;0-C;4;3;0;0;1      6
4     HIS  C;4;1;3;0;0-C;4;3;0;0;1     42


In [10]:
import numpy as np
import pandas as pd
from Bio import PDB
from rdkit import Chem
from rdkit.Chem import rdchem

# Function to calculate Euclidean distance
def euclidean_distance(coord1, coord2):
    return np.linalg.norm(np.array(coord1) - np.array(coord2))

# Load Protein PDB file
def load_protein_atoms(pdb_file):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    protein_atoms = []
    
    for model in structure:
        for chain in model:
            for residue in chain:
                if PDB.is_aa(residue):  # Consider only amino acid residues
                    res_name = residue.get_resname()
                    for atom in residue:
                        protein_atoms.append((res_name, atom.get_name(), atom.coord))
    
    return protein_atoms

# Load Ligand SDF file
def load_ligand_atoms(sdf_file):
    ligand_atoms = []
    suppl = Chem.SDMolSupplier(sdf_file)
    mol = suppl[0]  # Assume one ligand in SDF file
    
    for atom in mol.GetAtoms():
        pos = mol.GetConformer().GetAtomPosition(atom.GetIdx())
        ligand_atoms.append((atom.GetSymbol(), (pos.x, pos.y, pos.z)))
    
    return ligand_atoms

# Function to compute ECIF pairs
def compute_ecif_pairs(protein_atoms, ligand_atoms, distance_cutoff=6.0):
    ecif_data = []
    
    for res_name, prot_atom_name, prot_coord in protein_atoms:
        for lig_atom, lig_coord in ligand_atoms:
            dist = euclidean_distance(prot_coord, lig_coord)
            if dist <= distance_cutoff:
                ecif_pair = f"{prot_atom_name}-{lig_atom}"
                ecif_data.append([res_name, ecif_pair, round(dist, 2)])
    
    return ecif_data

# Main execution
pdb_file = r"E:\NewProject\example_ref\1a1e\1a1e_protein.pdb"   # Change to your PDB file path
sdf_file = r"E:\NewProject\example_ref\1a1e\1a1e_ligandMS.sdf"

protein_atoms = load_protein_atoms(pdb_file)
ligand_atoms = load_ligand_atoms(sdf_file)

ecif_pairs = compute_ecif_pairs(protein_atoms, ligand_atoms)

# Convert to DataFrame
df = pd.DataFrame(ecif_pairs, columns=["Residue", "Protein-ligand_ecif_pair", "Distance(Å)"])

# Count occurrences
df_counts = df.groupby(["Residue", "Protein-ligand_ecif_pair"]).size().reset_index(name="Count")

In [14]:
# # Save results
df.to_csv("ECIF_Pairs.csv", index=False)
df_counts.to_csv("ECIF_Pair_Counts.csv", index=False)


In [12]:
df.head()  # Print first few ECIF pairs

,Residue,Protein-ligand_ecif_pair,Distance(Å)
0,ILE,O-O,5.96
1,THR,HA-O,5.35
2,THR,C-O,5.88
3,ARG,N-P,5.70
4,ARG,N-O,5.12


In [13]:
df_counts.head() # Print first few ECIF pair counts

,Residue,Protein-ligand_ecif_pair,Count
0,ALA,HB2-O,1
1,ARG,1HH1-C,2
2,ARG,1HH1-N,1
3,ARG,1HH1-O,5
4,ARG,1HH1-P,1
